# IMPLEMENTACIÓN COMPLETA: RES2NET-SE(2) UNET Y ABLATION STUDIES

A continuación se presenta el notebook completo para entrenar y evaluar el modelo propuesto **Res2Net‑SE(2) UNet** y sus variantes de ablación sobre el dataset BRISC 2025. El diseño sigue exactamente la misma estructura y condiciones experimentales que los benchmarks previos, garantizando una comparación justa.

## Contexto y justificación del modelo

El análisis de los benchmarks sobre BRISC 2025 mostró que:

- Los cinco modelos clásicos (U‑Net, Attention U‑Net, TransUNet, UNet++, DeepLabV3+) alcanzan resultados muy similares, con diferencias dentro del ruido experimental.
- El rendimiento no es homogéneo: **Glioma** es la clase más difícil (mIoU ~68%) y **Meningioma** la más fácil (~91%).
- El **plano axial** es sistemáticamente más desafiante que el sagital.
- La combinación **Glioma × Axial** concentra la mayor dificultad.

Estos hallazgos indican que **no es suficiente aumentar la complejidad arquitectónica**; es necesario atacar explícitamente dos fuentes de variabilidad:

1. **Variabilidad morfológica inter‑tumor** (glioma difuso vs meningioma compacto vs pituitaria pequeño).
2. **Variabilidad por plano anatómico** (axial, coronal, sagital).

Para ello se propone una arquitectura híbrida original: **Res2Net‑SE(2) UNet**, que combina:

- **Res2Net** en el encoder → jerarquías de escalas dentro de cada bloque residual, capturando simultáneamente texturas finas y formas globales. Esto mejora la discriminación entre tipos tumorales.
- **Equivarianza roto‑traslacional SE(2)** → las operaciones son equivariantes a rotaciones y traslaciones mediante *group convolutions*. Así, un tumor aparece con representaciones consistentes independientemente del plano de adquisición, atacando directamente la variabilidad axial‑sagital‑coronal.
- **Decoder con Group Deconvolutions** → mantiene la consistencia geométrica durante el upsampling.
- **Pérdida combinada Dice + BCE** (la misma que en benchmarks) para manejar el desbalance extremo fondo/tumor.

Además del modelo completo, se entrenan dos variantes de ablación para medir la contribución individual de cada componente:

- **Res2NetUNet** → solo la parte multi‑escala (sin equivarianza).
- **SE2UNet** → solo la parte equivariante (sin estructura Res2Net).
- **Res2NetSE2UNet** → el modelo híbrido completo.

Todos los experimentos se ejecutan bajo las **mismas condiciones** que los benchmarks: mismas particiones, preprocesamiento, augmentación, optimizador (AdamW), scheduler (ReduceLROnPlateau), early stopping (paciencia 80) y mixed precision.



## 1. CONFIGURACIÓN DEL ENTORNO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q e2cnn==1.0.5
!pip install -q segmentation-models-pytorch==0.5.0
!pip install -q albumentations==1.4.0

import os
import gc
import re
import time
import copy
import json
import math
import random
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# e2cnn para group equivariant convolutions
from e2cnn import gspaces
from e2cnn import nn as enn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print("=" * 70)
print("CONFIGURACIÓN DEL ENTORNO — MODELO PROPIO Y ABLATION")
print("=" * 70)
print(f"PyTorch:     {torch.__version__}")
print(f"e2cnn:       {enn.__version__ if hasattr(enn, '__version__') else '1.0.5'}")
print(f"Device:      {device}")
if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Seed:        {SEED}")
print("=" * 70)

## 2. CONFIGURACIÓN GLOBAL Y PARÁMETROS

In [ ]:
# ============================================================
# 2. CONFIGURACIÓN GLOBAL Y PARÁMETROS
# ============================================================

IMG_SIZE = 512
BATCH_SIZE = 64
NUM_WORKERS = 8
PIN_MEMORY = True

N_ROTATIONS = 8
BASE_CHANNELS = 32
IN_CHANNELS = 1
NUM_CLASSES = 2

HPARAMS = {
    'max_epochs': 500,
    'patience': 80,
    'learning_rate': 2e-4,
    'weight_decay': 1e-4,
    'scheduler_patience': 20,
    'scheduler_factor': 0.5,
    'min_lr': 1e-7,
    'grad_clip': 1.0,
}

r2_act = gspaces.Rot2dOnR2(N=N_ROTATIONS)

print("Configuración global:")
print(f"  IMG_SIZE:        {IMG_SIZE}×{IMG_SIZE}")
print(f"  BATCH_SIZE:      {BATCH_SIZE}")
print(f"  N_ROTATIONS:     {N_ROTATIONS}")
print(f"  BASE_CHANNELS:   {BASE_CHANNELS}")
print("Hiperparámetros:")
for k, v in HPARAMS.items():
    print(f"    {k}: {v}")

## 3. CARGA DE DATOS Y DATALOADERS


In [ ]:
# ============================================================
# 3. CARGA DE DATOS Y DATALOADERS
# ============================================================

DRIVE_ROOT   = "/content/drive/MyDrive/tumores_ceb_proyecto"
SPLITS_DIR   = os.path.join(DRIVE_ROOT, "splits")
CKPT_DIR     = os.path.join(DRIVE_ROOT, "checkpoints_proposed")
RESULTS_DIR  = os.path.join(DRIVE_ROOT, "results_proposed")
FIGURES_DIR  = os.path.join(DRIVE_ROOT, "figures_proposed")
COLAB_DATA   = "/content/brisc2025"
ZIP_PATH     = os.path.join(DRIVE_ROOT, "brisc2025.zip")

for d in [CKPT_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(COLAB_DATA):
    print("Descomprimiendo dataset...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall("/content/")
    print(f"Completado en {(time.time()-t0)/60:.2f} min")
else:
    print(f"Dataset disponible en {COLAB_DATA}")

PLANE_TO_IDX = {'Axial': 0, 'Coronal': 1, 'Sagittal': 2}

df_train = pd.read_csv(os.path.join(SPLITS_DIR, 'split_train.csv'))
df_val   = pd.read_csv(os.path.join(SPLITS_DIR, 'split_val.csv'))
df_test  = pd.read_csv(os.path.join(SPLITS_DIR, 'split_test.csv'))

print(f"\nSplits cargados:")
print(f"  Train: {len(df_train)}")
print(f"  Val:   {len(df_val)}")
print(f"  Test:  {len(df_test)}")

class BRISCSegDataset(Dataset):
    def __init__(self, dataframe, transforms=None):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.array(Image.open(row['image_path']).convert('L'), dtype=np.float32) / 255.0
        mask = np.array(Image.open(row['mask_path']).convert('L'), dtype=np.float32) / 255.0
        mask = (mask > 0.5).astype(np.float32)

        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask  = transformed['mask']
        else:
            image = torch.from_numpy(image).float()
            mask  = torch.from_numpy(mask).float()

        image = image.float().clamp(0.0, 1.0)
        mask  = (mask > 0.5).float()

        if image.ndim == 2:
            image = image.unsqueeze(0)
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        meta = {
            'filename': row['filename'],
            'tumor_label': row['tumor_label'],
            'plane_label': row['plane_label'],
            'plane_idx': torch.tensor(PLANE_TO_IDX.get(row['plane_label'], 0), dtype=torch.long),
        }
        return image, mask, meta

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=2),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=10, border_mode=0, p=0.30),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.25),
    A.GaussNoise(var_limit=(3.0, 12.0), p=0.15),
    ToTensorV2(),
])

val_test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=2),
    ToTensorV2(),
])

# Datasets
train_dataset = BRISCSegDataset(df_train, transforms=train_transforms)
val_dataset   = BRISCSegDataset(df_val,   transforms=val_test_transforms)
test_dataset  = BRISCSegDataset(df_test,  transforms=val_test_transforms)

# DataLoaders
_dl_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **_dl_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, drop_last=False, **_dl_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False, drop_last=False, **_dl_kwargs)

print(f"\nDataLoaders creados:")
print(f"  Train: {len(train_dataset)} muestras → {len(train_loader)} batches")
print(f"  Val:   {len(val_dataset)} muestras → {len(val_loader)} batches")
print(f"  Test:  {len(test_dataset)} muestras → {len(test_loader)} batches")

images, masks, metas = next(iter(train_loader))
print(f"\nVerificación batch:")
print(f"  Images shape: {images.shape}")
print(f"  Masks unique: {torch.unique(masks).tolist()}")

## 4. FUNCIONES DE PÉRDIDA Y MÉTRICAS

In [ ]:
# ============================================================
# 4. FUNCIONES DE PÉRDIDA Y MÉTRICAS
# ============================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred_flat = pred.reshape(-1)
        target_flat = target.reshape(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        return 1.0 - dice

class ComboLoss(nn.Module):
    def __init__(self, alpha=0.5, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.dice_loss = DiceLoss(smooth=smooth)
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        return self.alpha * self.dice_loss(pred, target) + (1 - self.alpha) * self.bce_loss(pred, target)

def compute_iou(pred, target, threshold=0.5, smooth=1e-6):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - intersection
    return (intersection + smooth) / (union + smooth)

def compute_dice(pred, target, threshold=0.5, smooth=1e-6):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum(dim=(1,2,3))
    return (2.0 * intersection + smooth) / (pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + smooth)

print("Funciones de pérdida y métricas definidas (ComboLoss: Dice+BCE).")

## 5. MOTOR DE ENTRENAMIENTO

In [ ]:
# ============================================================
# 5. MOTOR DE ENTRENAMIENTO GENÉRICO
# ============================================================

AMP_ENABLED = torch.cuda.is_available()
AMP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, grad_clip=1.0):
    model.train()
    running_loss, running_iou, running_dice, n_batches = 0.0, 0.0, 0.0, 0
    for images, masks, _ in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            outputs = model(images)
            loss = criterion(outputs, masks)
        scaler.scale(loss).backward()
        if grad_clip is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        with torch.no_grad():
            running_loss += loss.item()
            running_iou += compute_iou(outputs, masks).mean().item()
            running_dice += compute_dice(outputs, masks).mean().item()
            n_batches += 1
    return {'loss': running_loss/n_batches, 'iou': running_iou/n_batches, 'dice': running_dice/n_batches}

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, running_iou, running_dice, n_batches = 0.0, 0.0, 0.0, 0
    for images, masks, _ in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            outputs = model(images)
            loss = criterion(outputs, masks)
        running_loss += loss.item()
        running_iou += compute_iou(outputs, masks).mean().item()
        running_dice += compute_dice(outputs, masks).mean().item()
        n_batches += 1
    return {'loss': running_loss/n_batches, 'iou': running_iou/n_batches, 'dice': running_dice/n_batches}

def train_model(model, model_name, train_loader, val_loader, device, hparams=HPARAMS):
    print(f"\n{'='*70}\nENTRENAMIENTO: {model_name}\n{'='*70}")
    total_p = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parámetros totales: {total_p:,} | Entrenables: {trainable_p:,} | Memoria: {total_p*4/1024**2:.1f} MB")

    model = model.to(device)
    criterion = ComboLoss(alpha=0.5)
    optimizer = optim.AdamW(model.parameters(), lr=hparams['learning_rate'], weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=hparams['scheduler_factor'],
                                                     patience=hparams['scheduler_patience'], min_lr=hparams['min_lr'])
    scaler = GradScaler(device='cuda', enabled=AMP_ENABLED)

    history = {'train_loss':[], 'train_iou':[], 'train_dice':[], 'val_loss':[], 'val_iou':[], 'val_dice':[], 'lr':[]}
    best_val_dice = -1.0
    best_model_state = None
    best_epoch = 0
    patience_counter = 0
    start_time = time.time()
    ckpt_path = os.path.join(CKPT_DIR, f'{model_name}_best.pth')

    for epoch in range(1, hparams['max_epochs']+1):
        epoch_start = time.time()
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, grad_clip=hparams['grad_clip'])
        val_metrics = evaluate(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['dice'])

        for k in ['loss','iou','dice']:
            history[f'train_{k}'].append(train_metrics[k])
            history[f'val_{k}'].append(val_metrics[k])
        history['lr'].append(current_lr)

        improved = val_metrics['dice'] > best_val_dice
        if improved:
            best_val_dice = val_metrics['dice']
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
            torch.save({'epoch':epoch, 'model_state_dict':best_model_state, 'val_dice':best_val_dice,
                        'val_iou':val_metrics['iou'], 'hparams':hparams}, ckpt_path)
        else:
            patience_counter += 1

        print(f"Epoch {epoch:03d}/{hparams['max_epochs']} | Train Loss: {train_metrics['loss']:.4f} | "
              f"Train IoU: {train_metrics['iou']:.4f} | Train Dice: {train_metrics['dice']:.4f} || "
              f"Val Loss: {val_metrics['loss']:.4f} | Val IoU: {val_metrics['iou']:.4f} | "
              f"Val Dice: {val_metrics['dice']:.4f} | LR: {current_lr:.2e} | "
              f"{time.time()-epoch_start:.1f}s | {'BEST' if improved else f'patience {patience_counter}/{hparams["patience"]}'}")

        if patience_counter >= hparams['patience']:
            print(f"\nEarly Stopping en epoch {epoch}. Mejor epoch: {best_epoch} | Val Dice: {best_val_dice:.4f}")
            break

    model.load_state_dict(best_model_state)
    with open(os.path.join(RESULTS_DIR, f"{model_name.lower().replace(' ', '_').replace('+','plus')}_history.json"), 'w') as f:
        json.dump(history, f, indent=2)

    total_time = time.time()-start_time
    print(f"\nResumen — {model_name}: mejor epoch {best_epoch}, Val Dice {best_val_dice:.4f}, tiempo {total_time/60:.1f} min")
    return model, history

print("Motor de entrenamiento definido.")

## 6. EVALUACIÓN DETALLADA EN TEST

In [ ]:
# ============================================================
# 6. EVALUACIÓN DETALLADA EN TEST
# ============================================================

@torch.no_grad()
def evaluate_detailed(model, loader, device, model_name="Model"):
    model.eval()
    all_results = []
    for images, masks, metas in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            outputs = model(images)
        ious = compute_iou(outputs, masks)
        dices = compute_dice(outputs, masks)
        for i in range(len(images)):
            all_results.append({
                'filename': metas['filename'][i],
                'tumor_label': metas['tumor_label'][i],
                'plane_label': metas['plane_label'][i],
                'iou': ious[i].item(),
                'dice': dices[i].item(),
            })
    results_df = pd.DataFrame(all_results)

    tumor_iou = results_df.groupby('tumor_label')['iou'].mean()
    tumor_counts = results_df.groupby('tumor_label')['iou'].count()
    weighted_miou = (tumor_iou * tumor_counts / tumor_counts.sum()).sum()
    plane_iou = results_df.groupby('plane_label')['iou'].mean()
    global_iou = results_df['iou'].mean()
    global_dice = results_df['dice'].mean()

    summary = {
        'model_name': model_name,
        'global_iou': global_iou,
        'global_dice': global_dice,
        'weighted_miou': weighted_miou,
        'tumor_iou': tumor_iou.to_dict(),
        'plane_iou': plane_iou.to_dict(),
        'tumor_dice': results_df.groupby('tumor_label')['dice'].mean().to_dict(),
        'plane_dice': results_df.groupby('plane_label')['dice'].mean().to_dict(),
    }

    print(f"\n{'='*70}\nRESULTADOS EN TEST — {model_name}\n{'='*70}")
    print(f"Weighted mIoU: {weighted_miou*100:.2f}%")
    print(f"Global mIoU:   {global_iou*100:.2f}%")
    print(f"Global Dice:   {global_dice*100:.2f}%")
    print("\nmIoU por Tipo Tumoral:")
    for t in ['Glioma','Meningioma','Pituitary']:
        print(f"  {t:<15} {tumor_iou.get(t,0)*100:.2f}%")
    print("\nmIoU por Plano Anatómico:")
    for p in ['Axial','Coronal','Sagittal']:
        print(f"  {p:<15} {plane_iou.get(p,0)*100:.2f}%")
    return results_df, summary

def save_model_results(model_name, model, history, test_loader, device):
    safe_name = model_name.lower().replace(' ', '_').replace('+','plus')
    results_df, summary = evaluate_detailed(model, test_loader, device, model_name)
    results_df.to_csv(os.path.join(RESULTS_DIR, f'{safe_name}_test_results.csv'), index=False)
    with open(os.path.join(RESULTS_DIR, f'{safe_name}_summary.json'), 'w') as f:
        json.dump({k:v for k,v in summary.items() if not callable(v)}, f, indent=2)
    print(f"\nResultados guardados: {safe_name}_test_results.csv y {safe_name}_summary.json")
    return results_df, summary

def model_already_trained(model_name):
    safe = model_name.lower().replace(' ', '_').replace('+','plus')
    ckpt = os.path.join(CKPT_DIR, f'{model_name}_best.pth')
    summ = os.path.join(RESULTS_DIR, f'{safe}_summary.json')
    hist = os.path.join(RESULTS_DIR, f'{safe}_history.json')
    exists = all(os.path.exists(p) for p in [ckpt, summ, hist])
    if exists:
        ck = torch.load(ckpt, map_location='cpu')
        print(f"  {model_name} encontrado — Val Dice: {ck['val_dice']:.4f} (epoch {ck['epoch']})")
    return exists

def load_saved_results(model_name):
    safe = model_name.lower().replace(' ', '_').replace('+','plus')
    with open(os.path.join(RESULTS_DIR, f'{safe}_history.json'), 'r') as f:
        history = json.load(f)
    with open(os.path.join(RESULTS_DIR, f'{safe}_summary.json'), 'r') as f:
        summary = json.load(f)
    results_df = pd.read_csv(os.path.join(RESULTS_DIR, f'{safe}_test_results.csv'))
    print(f"  Resultados cargados: {len(results_df)} imágenes, {len(history['train_loss'])} epochs")
    return history, summary, results_df

print("Funciones de evaluación definidas.")

## 7. ARQUITECTURAS PARA ABLATION

### 7.1. Solo Res2Net (sin equivarianza) – Res2NetUNet

In [ ]:
# ============================================================
# 7.1. RES2NET-UNET (SIN EQUIVARIANZA)
# ============================================================

class Res2NetBlock(nn.Module):
    """Bloque Res2Net estándar (convoluciones normales, no group)"""
    def __init__(self, in_ch, out_ch, scales=4, stride=1):
        super().__init__()
        self.scales = scales
        width = out_ch // scales

        self.entry_conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.entry_bn = nn.BatchNorm2d(out_ch)
        self.entry_relu = nn.ReLU(inplace=True)

        self.sub_convs = nn.ModuleList([
            nn.Conv2d(width, width, 3, padding=1, stride=stride, bias=False)
            for _ in range(scales-1)
        ])
        self.sub_bns = nn.ModuleList([nn.BatchNorm2d(width) for _ in range(scales-1)])
        self.sub_relu = nn.ReLU(inplace=True)

        self.exit_conv = nn.Conv2d(out_ch, out_ch, 1, bias=False)
        self.exit_bn = nn.BatchNorm2d(out_ch)

        if in_ch != out_ch or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = None

        self.out_relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x) if self.shortcut else x
        out = self.entry_relu(self.entry_bn(self.entry_conv(x)))
        chunks = torch.chunk(out, self.scales, dim=1)
        processed = []
        sp = None
        for i, chunk in enumerate(chunks):
            if i == 0:
                processed.append(chunk)
            elif i < self.scales-1:
                if sp is not None:
                    chunk = chunk + sp
                sp = self.sub_relu(self.sub_bns[i-1](self.sub_convs[i-1](chunk)))
                processed.append(sp)
            else:
                processed.append(chunk)
        out = torch.cat(processed, dim=1)
        out = self.exit_bn(self.exit_conv(out))
        out = self.out_relu(out + residual)
        return out

class Res2NetEncoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, base_channels, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(base_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(2, stride=2)
        self.enc1 = Res2NetBlock(base_channels, base_channels*2, stride=2)
        self.enc2 = Res2NetBlock(base_channels*2, base_channels*4, stride=2)
        self.enc3 = Res2NetBlock(base_channels*4, base_channels*8, stride=2)
        self.enc4 = Res2NetBlock(base_channels*8, base_channels*16, stride=2)

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))          # 1/2
        x_pool = self.maxpool(x0)                        # 1/4
        s1 = self.enc1(x_pool)                           # 1/4
        s2 = self.enc2(s1)                               # 1/8
        s3 = self.enc3(s2)                               # 1/16
        s4 = self.enc4(s3)                               # 1/32
        return s4, [x0, s1, s2, s3]

class Res2NetDecoder(nn.Module):
    def __init__(self, num_classes=2, base_channels=32):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(base_channels*16, base_channels*8, 2, stride=2)
        self.dec4 = self._double_conv(base_channels*8 + base_channels*8, base_channels*8)
        self.up3 = nn.ConvTranspose2d(base_channels*8, base_channels*4, 2, stride=2)
        self.dec3 = self._double_conv(base_channels*4 + base_channels*4, base_channels*4)
        self.up2 = nn.ConvTranspose2d(base_channels*4, base_channels*2, 2, stride=2)
        self.dec2 = self._double_conv(base_channels*2 + base_channels*2, base_channels*2)
        self.up1 = nn.ConvTranspose2d(base_channels*2, base_channels, 2, stride=2)
        self.dec1 = self._double_conv(base_channels + base_channels, base_channels)
        self.final = nn.Conv2d(base_channels, num_classes, 1)

    def _double_conv(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, bottleneck, skips):
        x0, s1, s2, s3 = skips
        d4 = self.up4(bottleneck)
        d4 = self.dec4(torch.cat([d4, s3], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, s2], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, s1], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, x0], dim=1))
        return self.final(d1)

class Res2NetUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, base_channels=32):
        super().__init__()
        self.encoder = Res2NetEncoder(in_channels, base_channels)
        self.decoder = Res2NetDecoder(num_classes, base_channels)

    def forward(self, x):
        bottleneck, skips = self.encoder(x)
        logits = self.decoder(bottleneck, skips)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return logits

def build_res2net_unet():
    return Res2NetUNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES, base_channels=BASE_CHANNELS)

print("Res2NetUNet (solo multi‑escala) definido.")

### 7.2. Solo equivarianza SE(2) – SE2UNet

In [ ]:
# ============================================================
# 7.2. SE2-UNET (SOLO EQUIVARIANZA, SIN RES2NET)
# ============================================================

class SE2Encoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=32):
        super().__init__()
        self.in_type = enn.FieldType(r2_act, in_channels * [r2_act.trivial_repr])
        self.lifting = enn.R2Conv(self.in_type,
                                  enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]),
                                  kernel_size=7, padding=3, bias=False)
        self.lifting_bn = enn.InnerBatchNorm(enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]))
        self.lifting_relu = enn.ReLU(enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]), inplace=True)

        # Tipos
        self.type1 = enn.FieldType(r2_act, base_channels * [r2_act.regular_repr])
        self.type2 = enn.FieldType(r2_act, (base_channels*2) * [r2_act.regular_repr])
        self.type3 = enn.FieldType(r2_act, (base_channels*4) * [r2_act.regular_repr])
        self.type4 = enn.FieldType(r2_act, (base_channels*8) * [r2_act.regular_repr])
        self.type5 = enn.FieldType(r2_act, (base_channels*16) * [r2_act.regular_repr])

        # Bloques convolucionales simples (sin estructura Res2Net)
        self.conv1 = enn.R2Conv(self.type1, self.type2, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = enn.InnerBatchNorm(self.type2)
        self.relu1 = enn.ReLU(self.type2, inplace=True)

        self.conv2 = enn.R2Conv(self.type2, self.type3, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn2 = enn.InnerBatchNorm(self.type3)
        self.relu2 = enn.ReLU(self.type3, inplace=True)

        self.conv3 = enn.R2Conv(self.type3, self.type4, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn3 = enn.InnerBatchNorm(self.type4)
        self.relu3 = enn.ReLU(self.type4, inplace=True)

        self.conv4 = enn.R2Conv(self.type4, self.type5, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn4 = enn.InnerBatchNorm(self.type5)
        self.relu4 = enn.ReLU(self.type5, inplace=True)

        self.pool = enn.PointwiseMaxPool(self.type1, kernel_size=2, stride=2)

    def forward(self, x):
        geo_x = enn.GeometricTensor(x, self.in_type)
        lifted = self.lifting_relu(self.lifting_bn(self.lifting(geo_x)))
        s1 = lifted
        pooled = self.pool(s1)
        s2 = self.relu1(self.bn1(self.conv1(pooled)))
        s3 = self.relu2(self.bn2(self.conv2(s2)))
        s4 = self.relu3(self.bn3(self.conv3(s3)))
        bot = self.relu4(self.bn4(self.conv4(s4)))
        return bot, [s1, s2, s3, s4]

class SE2Decoder(nn.Module):
    def __init__(self, num_classes=2, base_channels=32):
        super().__init__()
        N = N_ROTATIONS
        self.type5 = enn.FieldType(r2_act, (base_channels*16) * [r2_act.regular_repr])
        self.type4 = enn.FieldType(r2_act, (base_channels*8) * [r2_act.regular_repr])
        self.type3 = enn.FieldType(r2_act, (base_channels*4) * [r2_act.regular_repr])
        self.type2 = enn.FieldType(r2_act, (base_channels*2) * [r2_act.regular_repr])
        self.type1 = enn.FieldType(r2_act, base_channels * [r2_act.regular_repr])

        self.up4 = enn.R2ConvTransposed(self.type5, self.type4, kernel_size=2, stride=2, bias=False)
        self.bn4 = enn.InnerBatchNorm(self.type4)
        self.relu4 = enn.ReLU(self.type4, inplace=True)
        self.dec4 = self._double_conv(self.type4, self.type4, self.type4)

        self.up3 = enn.R2ConvTransposed(self.type4, self.type3, kernel_size=2, stride=2, bias=False)
        self.bn3 = enn.InnerBatchNorm(self.type3)
        self.relu3 = enn.ReLU(self.type3, inplace=True)
        self.dec3 = self._double_conv(self.type3, self.type3, self.type3)

        self.up2 = enn.R2ConvTransposed(self.type3, self.type2, kernel_size=2, stride=2, bias=False)
        self.bn2 = enn.InnerBatchNorm(self.type2)
        self.relu2 = enn.ReLU(self.type2, inplace=True)
        self.dec2 = self._double_conv(self.type2, self.type2, self.type2)

        self.up1 = enn.R2ConvTransposed(self.type2, self.type1, kernel_size=2, stride=2, bias=False)
        self.bn1 = enn.InnerBatchNorm(self.type1)
        self.relu1 = enn.ReLU(self.type1, inplace=True)
        self.dec1 = self._double_conv(self.type1, self.type1, self.type1)

        self.group_pool = enn.GroupPooling(self.type1)
        self.head = nn.Conv2d(base_channels, num_classes, 1)

    def _double_conv(self, in_type, mid_type, out_type):
        return nn.Sequential(
            enn.R2Conv(in_type, mid_type, kernel_size=3, padding=1, bias=False),
            enn.InnerBatchNorm(mid_type),
            enn.ReLU(mid_type, inplace=True),
            enn.R2Conv(mid_type, out_type, kernel_size=3, padding=1, bias=False),
            enn.InnerBatchNorm(out_type),
            enn.ReLU(out_type, inplace=True)
        )

    def _cat(self, up, skip):
        fuse_ch = up.tensor.shape[1] + skip.tensor.shape[1]
        fuse_type = enn.FieldType(r2_act, (fuse_ch // N_ROTATIONS) * [r2_act.regular_repr])
        return enn.GeometricTensor(torch.cat([up.tensor, skip.tensor], dim=1), fuse_type)

    def forward(self, bottleneck, skips):
        s1, s2, s3, s4 = skips
        x = self.relu4(self.bn4(self.up4(bottleneck)))
        x = self.dec4(self._cat(x, s4))
        x = self.relu3(self.bn3(self.up3(x)))
        x = self.dec3(self._cat(x, s3))
        x = self.relu2(self.bn2(self.up2(x)))
        x = self.dec2(self._cat(x, s2))
        x = self.relu1(self.bn1(self.up1(x)))
        x = self.dec1(self._cat(x, s1))
        x = self.group_pool(x)
        return self.head(x.tensor)

class SE2UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, base_channels=32):
        super().__init__()
        self.encoder = SE2Encoder(in_channels, base_channels)
        self.decoder = SE2Decoder(num_classes, base_channels)

    def forward(self, x):
        bottleneck, skips = self.encoder(x)
        logits = self.decoder(bottleneck, skips)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return logits

def build_se2_unet():
    return SE2UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES, base_channels=BASE_CHANNELS)

print("SE2UNet (solo equivarianza) definido.")

### 7.3. Modelo completo: Res2Net‑SE(2) UNet

In [ ]:
# ============================================================
# 7.3. RES2NET-SE(2) UNET (COMPLETO)
# ============================================================

class EquivariantRes2NetBlock(nn.Module):
    """Bloque Res2Net con group convolutions equivariantes SE(2)"""
    def __init__(self, in_type, out_type, scales=4, stride=1):
        super().__init__()
        self.scales = scales
        self.out_type = out_type
        N = N_ROTATIONS
        width = out_type.size // scales
        self.sub_type = enn.FieldType(r2_act, (width // N) * [r2_act.regular_repr])

        self.entry_conv = enn.R2Conv(in_type, out_type, kernel_size=1, bias=False)
        self.entry_bn = enn.InnerBatchNorm(out_type)
        self.entry_relu = enn.ReLU(out_type, inplace=True)

        self.sub_convs = nn.ModuleList([
            enn.R2Conv(self.sub_type, self.sub_type, kernel_size=3, padding=1, stride=stride, bias=False)
            for _ in range(scales-1)
        ])
        self.sub_bns = nn.ModuleList([enn.InnerBatchNorm(self.sub_type) for _ in range(scales-1)])
        self.sub_relu = enn.ReLU(self.sub_type, inplace=True)

        self.exit_conv = enn.R2Conv(out_type, out_type, kernel_size=1, bias=False)
        self.exit_bn = enn.InnerBatchNorm(out_type)

        if in_type != out_type or stride != 1:
            self.shortcut = nn.Sequential(
                enn.R2Conv(in_type, out_type, kernel_size=1, stride=stride, bias=False),
                enn.InnerBatchNorm(out_type)
            )
        else:
            self.shortcut = None
        self.out_relu = enn.ReLU(out_type, inplace=True)

    def forward(self, x):
        residual = self.shortcut(x) if self.shortcut else x
        out = self.entry_relu(self.entry_bn(self.entry_conv(x)))
        chunks = torch.chunk(out.tensor, self.scales, dim=1)
        processed = []
        sp = None
        for i, chunk in enumerate(chunks):
            geo = enn.GeometricTensor(chunk, self.sub_type)
            if i == 0:
                processed.append(geo)
            elif i < self.scales-1:
                if sp is not None:
                    geo = enn.GeometricTensor(chunk + sp.tensor, self.sub_type)
                sp = self.sub_relu(self.sub_bns[i-1](self.sub_convs[i-1](geo)))
                processed.append(sp)
            else:
                processed.append(geo)
        cat = torch.cat([p.tensor for p in processed], dim=1)
        out = self.exit_bn(self.exit_conv(enn.GeometricTensor(cat, self.out_type)))
        fused = enn.GeometricTensor(out.tensor + residual.tensor, self.out_type)
        return self.out_relu(fused)

class Res2NetSE2Encoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=32):
        super().__init__()
        self.in_type = enn.FieldType(r2_act, in_channels * [r2_act.trivial_repr])
        self.lifting = enn.R2Conv(self.in_type,
                                  enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]),
                                  kernel_size=7, padding=3, bias=False)
        self.lifting_bn = enn.InnerBatchNorm(enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]))
        self.lifting_relu = enn.ReLU(enn.FieldType(r2_act, base_channels * [r2_act.regular_repr]), inplace=True)

        self.type1 = enn.FieldType(r2_act, base_channels * [r2_act.regular_repr])
        self.type2 = enn.FieldType(r2_act, (base_channels*2) * [r2_act.regular_repr])
        self.type3 = enn.FieldType(r2_act, (base_channels*4) * [r2_act.regular_repr])
        self.type4 = enn.FieldType(r2_act, (base_channels*8) * [r2_act.regular_repr])
        self.type5 = enn.FieldType(r2_act, (base_channels*16) * [r2_act.regular_repr])

        self.pool = enn.PointwiseMaxPool(self.type1, kernel_size=2, stride=2)
        self.enc1 = EquivariantRes2NetBlock(self.type1, self.type2, stride=2)
        self.enc2 = EquivariantRes2NetBlock(self.type2, self.type3, stride=2)
        self.enc3 = EquivariantRes2NetBlock(self.type3, self.type4, stride=2)
        self.enc4 = EquivariantRes2NetBlock(self.type4, self.type5, stride=2)

    def forward(self, x):
        geo_x = enn.GeometricTensor(x, self.in_type)
        s1 = self.lifting_relu(self.lifting_bn(self.lifting(geo_x)))
        pooled = self.pool(s1)
        s2 = self.enc1(pooled)
        s3 = self.enc2(s2)
        s4 = self.enc3(s3)
        bot = self.enc4(s4)
        return bot, [s1, s2, s3, s4]

class GroupDeconvBlock(nn.Module):
    def __init__(self, in_type, skip_type, out_type):
        super().__init__()
        N = N_ROTATIONS
        self.up = enn.R2ConvTransposed(in_type, out_type, kernel_size=2, stride=2, bias=False)
        self.up_bn = enn.InnerBatchNorm(out_type)
        self.up_relu = enn.ReLU(out_type, inplace=True)
        fuse_ch = (out_type.size + skip_type.size) // N
        self.fuse_type = enn.FieldType(r2_act, fuse_ch * [r2_act.regular_repr])
        self.fuse_conv = enn.R2Conv(self.fuse_type, out_type, kernel_size=3, padding=1, bias=False)
        self.fuse_bn = enn.InnerBatchNorm(out_type)
        self.fuse_relu = enn.ReLU(out_type, inplace=True)

    def forward(self, x, skip):
        x = self.up_relu(self.up_bn(self.up(x)))
        fused = enn.GeometricTensor(torch.cat([x.tensor, skip.tensor], dim=1), self.fuse_type)
        return self.fuse_relu(self.fuse_bn(self.fuse_conv(fused)))

class Res2NetSE2Decoder(nn.Module):
    def __init__(self, num_classes=2, base_channels=32):
        super().__init__()
        N = N_ROTATIONS
        self.type5 = enn.FieldType(r2_act, (base_channels*16) * [r2_act.regular_repr])
        self.type4 = enn.FieldType(r2_act, (base_channels*8) * [r2_act.regular_repr])
        self.type3 = enn.FieldType(r2_act, (base_channels*4) * [r2_act.regular_repr])
        self.type2 = enn.FieldType(r2_act, (base_channels*2) * [r2_act.regular_repr])
        self.type1 = enn.FieldType(r2_act, base_channels * [r2_act.regular_repr])

        self.dec4 = GroupDeconvBlock(self.type5, self.type4, self.type4)
        self.dec3 = GroupDeconvBlock(self.type4, self.type3, self.type3)
        self.dec2 = GroupDeconvBlock(self.type3, self.type2, self.type2)
        self.dec1 = GroupDeconvBlock(self.type2, self.type1, self.type1)

        self.group_pool = enn.GroupPooling(self.type1)
        self.head = nn.Conv2d(base_channels, num_classes, 1)

    def forward(self, bottleneck, skips):
        s1, s2, s3, s4 = skips
        x = self.dec4(bottleneck, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.group_pool(x)
        return self.head(x.tensor)

class Res2NetSE2UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, base_channels=32):
        super().__init__()
        self.encoder = Res2NetSE2Encoder(in_channels, base_channels)
        self.decoder = Res2NetSE2Decoder(num_classes, base_channels)

    def forward(self, x):
        bottleneck, skips = self.encoder(x)
        logits = self.decoder(bottleneck, skips)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return logits

def build_res2net_se2_unet():
    return Res2NetSE2UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES, base_channels=BASE_CHANNELS)

print("Res2NetSE2UNet (completo) definido.")

## 8. ENTRENAMIENTO DE LAS VARIANTES (ABLATION)
Cada celda siguiente entrena una variante. Si el modelo ya existe en disco (checkpoint + resultados), se cargan directamente sin reentrenar.

### 8.1. Entrenar Res2NetUNet

In [ ]:
# ============================================================
# 8.1. ENTRENAR RES2NETUNET (Solo multi‑escala)
# ============================================================

_name = "Res2NetUNet"
_build_fn = build_res2net_unet

if model_already_trained(_name):
    print(f"\n{_name} ya entrenado. Cargando resultados...")
    _history, _summary, _results_df = load_saved_results(_name)
else:
    clear_memory()
    _model = _build_fn()
    _model, _history = train_model(_model, _name, train_loader, val_loader, device, HPARAMS)
    _results_df, _summary = save_model_results(_name, _model, _history, test_loader, device)
    del _model
    clear_memory()

# Guardar en diccionarios globales
all_summaries = {} if 'all_summaries' not in dir() else all_summaries
all_results = {} if 'all_results' not in dir() else all_results
all_histories = {} if 'all_histories' not in dir() else all_histories
all_summaries[_name] = _summary
all_results[_name] = _results_df
all_histories[_name] = _history
print(f"\n{_name} completado.")

### 8.2. Entrenar SE2UNet

In [ ]:
# ============================================================
# 8.2. ENTRENAR SE2UNET (Solo equivarianza)
# ============================================================

_name = "SE2UNet"
_build_fn = build_se2_unet

if model_already_trained(_name):
    print(f"\n{_name} ya entrenado. Cargando resultados...")
    _history, _summary, _results_df = load_saved_results(_name)
else:
    clear_memory()
    _model = _build_fn()
    _model, _history = train_model(_model, _name, train_loader, val_loader, device, HPARAMS)
    _results_df, _summary = save_model_results(_name, _model, _history, test_loader, device)
    del _model
    clear_memory()

all_summaries[_name] = _summary
all_results[_name] = _results_df
all_histories[_name] = _history
print(f"\n{_name} completado.")

### 8.3. Entrenar el modelo completo Res2NetSE2UNet

In [ ]:
# ============================================================
# 8.3. ENTRENAR RES2NETSE2UNET (Completo)
# ============================================================

_name = "Res2NetSE2UNet"
_build_fn = build_res2net_se2_unet

if model_already_trained(_name):
    print(f"\n{_name} ya entrenado. Cargando resultados...")
    _history, _summary, _results_df = load_saved_results(_name)
else:
    clear_memory()
    _model = _build_fn()
    _model, _history = train_model(_model, _name, train_loader, val_loader, device, HPARAMS)
    _results_df, _summary = save_model_results(_name, _model, _history, test_loader, device)
    del _model
    clear_memory()

all_summaries[_name] = _summary
all_results[_name] = _results_df
all_histories[_name] = _history
print(f"\n{_name} completado.")

## 9. TABLA COMPARATIVA CON BENCHMARKS Y ABLATION

In [ ]:
# ============================================================
# 9. TABLA COMPARATIVA CON BENCHMARKS Y ABLATION
# ============================================================

# Cargar resultados de benchmarks previos (deben estar en sus directorios)
benchmark_models = ['UNet', 'AttentionUNet', 'TransUNet', 'UNetPlusPlus', 'DeepLabV3Plus']
for bm in benchmark_models:
    # Usar la función de benchmarks (ajustar rutas si es necesario)
    ckpt_bench = os.path.join("/content/drive/MyDrive/tumores_ceb_proyecto/checkpoints_benchmark", f'{bm}_best.pth')
    if os.path.exists(ckpt_bench) and bm not in all_summaries:
        # Cargar desde resultados_benchmark
        safe = bm.lower().replace(' ', '_').replace('+','plus')
        hist_path = os.path.join("/content/drive/MyDrive/tumores_ceb_proyecto/results_benchmark", f'{safe}_history.json')
        summ_path = os.path.join("/content/drive/MyDrive/tumores_ceb_proyecto/results_benchmark", f'{safe}_summary.json')
        res_path = os.path.join("/content/drive/MyDrive/tumores_ceb_proyecto/results_benchmark", f'{safe}_test_results.csv')
        if os.path.exists(summ_path):
            with open(summ_path, 'r') as f:
                summary = json.load(f)
            all_summaries[bm] = summary
            print(f"Cargado benchmark {bm}")

# Construir DataFrame comparativo
rows = []
for model_name, summary in all_summaries.items():
    rows.append({
        'model': model_name,
        'glioma': summary['tumor_iou'].get('Glioma', 0) * 100,
        'meningioma': summary['tumor_iou'].get('Meningioma', 0) * 100,
        'pituitary': summary['tumor_iou'].get('Pituitary', 0) * 100,
        'w_miou': summary['weighted_miou'] * 100,
        'axial': summary['plane_iou'].get('Axial', 0) * 100,
        'coronal': summary['plane_iou'].get('Coronal', 0) * 100,
        'sagittal': summary['plane_iou'].get('Sagittal', 0) * 100,
        'dice_global': summary['global_dice'] * 100,
    })

df_compare = pd.DataFrame(rows).sort_values('w_miou', ascending=False).reset_index(drop=True)

print("\n" + "="*110)
print("TABLA COMPARATIVA: BENCHMARKS + NUESTRAS VARIANTES (ABLATION)")
print("="*110)
print(df_compare[['model','glioma','meningioma','pituitary','w_miou','axial','coronal','sagittal','dice_global']].to_string(index=False))
print("="*110)

# Guardar en CSV
df_compare.to_csv(os.path.join(RESULTS_DIR, 'ablation_comparison.csv'), index=False)
print(f"\nTabla guardada en {os.path.join(RESULTS_DIR, 'ablation_comparison.csv')}")

## 10. VISUALIZACIÓN DE CURVAS DE ENTRENAMIENTO

In [ ]:
# ============================================================
# 10. VISUALIZACIÓN DE CURVAS
# ============================================================

def plot_training_curves(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    epochs = range(1, len(history['train_loss'])+1)
    for ax, tr, vl, lbl in zip(axes, ['train_loss','train_iou','train_dice'],
                                ['val_loss','val_iou','val_dice'], ['Loss','IoU','Dice']):
        ax.plot(epochs, history[tr], label='Train')
        ax.plot(epochs, history[vl], label='Validation')
        ax.set_title(f'{model_name} — {lbl}')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'Curvas de entrenamiento — {model_name}')
    plt.tight_layout()
    plt.show()

if 'Res2NetSE2UNet' in all_histories:
    plot_training_curves(all_histories['Res2NetSE2UNet'], 'Res2Net-SE(2) UNet')